In [0]:
%run ./01-config

In [0]:
class Producer():
    def __init__(self):
        self.config = Config()
        self.landing_zone = self.config.base_dir_data + "/raw"
        self.test_data_dir = self.config.base_dir_data + "/test_data"
    
    def user_registration(self, set_num):
        source = f"{self.test_data_dir}/1-registered_users_{set_num}.csv"
        target = f"{self.landing_zone}/registered_users_bz/1-registered_users_{set_num}.csv"
        print(f"Producing {source}...", end='')
        dbutils.fs.cp(source, target)
        print("done")
    
    def profile_cdc(self, set_num):
        source = f"{self.test_data_dir}/2-user_info_{set_num}.json"
        target = f"{self.landing_zone}/kafka_multiplex_bz/2-user_info_{set_num}.json"
        print(f"Producing {source}...", end='')
        dbutils.fs.cp(source, target)
        print("done")
    
    def bpm(self, set_num):
        source = f"{self.test_data_dir}/3-bpm_{set_num}.json"
        target = f"{self.landing_zone}/kafka_multiplex_bz/3-bpm_{set_num}.json"
        print(f"Producing {source}...", end='')
        dbutils.fs.cp(source, target)
        print("done")
    
    def workout(self, set_num):
        source = f"{self.test_data_dir}/4-bpm_{set_num}.json"
        target = f"{self.landing_zone}/kafka_multiplex_bz/4-workout_{set_num}.json"
        print(f"Producing {source}...", end='')
        dbutils.fs.cp(source, target)
        print("done")
    
    def gym_logins(self, set_num):
        source = f"{self.test_data_dir}/5-gym_logins_{set_num}.csv"
        target = f"{self.landing_zone}/kafka_multiplex_bz/5-gym_logins_{set_num}.csv"
        print(f"Producing {source}...", end='')
        dbutils.fs.cp(source, target)
        print("done")
    
    def produce(self, set_num):
        import time
        start = int(time.time())
        print(f"Producing test data set {set_num}")
        self.user_registration(set_num)
        self.profile_cdc(set_num)
        self.bpm(set_num)
        self.workout(set_num)
        self.gym_logins(set_num)
        print(f"Test data set {set_num} produced in {int(time.time()) - start} seconds")

    def _validate_count(self, format, location, expected_count):
        print(f"Validating {location}...", end='')
        target = f"{self.landing_zone}/{location}_*.{format}"
        count = spark.read.format(format).option("header", "true").load(target).count()
        assert count == expected_count, f"Expected {expected_count} records in {location}, found {count}"
        print("Validation successful")

    def validate(self, set_num):
        import time
        start = int(time.time())
        print(f"Validating test data set {set_num}")
        self._validate_count("csv", f"registered_users_bz/1-registered_users_{set_num}", 5 if sets == 1 else 10)
        self._validate_count("json", f"kafka_multiplex_bz/2-user_info_{set_num}", 5 if sets == 1 else 10)
        self._validate_count("json", f"kafka_multiplex_bz/3-bpm_{set_num}", 5 if sets == 1 else 10)
        self._validate_count("json", f"kafka_multiplex_bz/4-workout_{set_num}", 5 if sets == 1 else 10)
        self._validate_count("csv", f"kafka_multiplex_bz/5-gym_logins_{set_num}", 5 if sets == 1 else 10)
        print(f"Test data set {set_num} validated in {int(time.time()) - start} seconds")
